프로젝트 주제: 
"배달 서비스 퀄리티 유지를 위해 라이더의 낮은 평점대 주문 분석 및 예측 모델링" 

1) 문제 정의 

- 2월 한 주 (2/11-2/18), 3월 (22일 미포함), 4월 한 주 (4/1-4/6)의 총 45000여건의 주문 중 총 140건의 주문의 라이더 평점이 3점대 이하
- 배달 서비스 퀄리티 유지를 위해 낮은 라이더 평점대의 요인을 파악하고 개선안 필요

2) 데이터 

배달 데이터 ('order_id', 'rider_id', 'rider_age',
       'rider_ratings', 'restaurant_latitude',
       'restaurant_longitude', 'delivery_location_latitude',
       'delivery_location_longitude', 'order_date', 'time_ordered',
       'time_picked', 'weather', 'traffic',
       'vehicle_condition', 'order_type', 'vehicle',
       'multiple_deliveries', 'festival', 'area_type', 'delivery_min')
       
(주문 아이디, 라이더 아이디, 라이더 나이, 라이더 평점, 상점 위도, 상점 경도, 배달 위도, 배달 경도, 주문 날짜, 주문 시간, 주문 픽업 시간, 날씨, 교통체증, 배달 수단 상태, 주문 종류, 배달 수단, 여러 주문 배달, 행사 주문 여부, 지역, 배달 시간(분))

약 2개월치 약 45000건 


3) 핵심 지표 정의

- 라이더 당 평균 평점: 4.3~ 4.9점
- 평점 3점 이하 평균 평점: 2.7점 
- 평점 3점 이하 평균 배달 시간: 37분 (맑은 날씨, 낮은 교통량)
- 평점 1점 평균 배달 시간: 26분 (안개 낀 날씨, 낮은 교통량)


4) 분석 과정

(1) EDA

- 총 43000여개의 주문 중 140개 주문의 라이더 평점이 3점 이하 
    - 105개의 라이더 평점 3점이하의 평균 평점 = 2.7점
    - 35개의 라이더 평점 = 1점
 
- 각 라이더의 평균 평점: 4.3 ~ 4.9점
  
- 140개 라이더 평점이 3점 이하
    - 피크타임(18 ~ 21시) 주문 아님:(22시~ 24시) 혹은 NaN
    - 낮은 교통체증 (Low Traffic)
        - 105개: Sunny (맑은 날씨)
        - 35개: Fog (안개 낀 날씨)
     
    - 105개의 맑은 날씨, 낮은 교통량 평균 배달 시간: 37분
    - 35개의 안개 낀 날씨, 낮은 교통량 평균 배달 시간: 26분
 
(2) 원인분석 

- 2347개의 맑은 날씨, 낮은 교통량의 "30분 초과 배달시간" 평점: 3.5점 이하 2.5 이상 
- 2347개의 맑은 날씨, 낮은 교통량의 "30분 이하 배달시간" 평점: 4.5 이상 
    - 배달시간 30분 이후를 기점으로 평점이 최저 4.5점에서 3.5점 이하대로 하락 

- 416개의 맑은 날씨, 낮은 교통량 평균 배달 시간 
    - 105개의 맑은 날씨, 낮은 교통량 (평점 3점 이하) 평균 배달 시간: 37분 
    - 311개의 맑은 날씨, 낮은 교통량 (평점 5점) 평균 배달 시간: 19분
      
(3) 개선 시뮬레이션 
- 415개의 맑은 날씨, 낮은 교통량의 "라이더 평점" 과 "배달 시간"의 관계를 모델링을 하여
    - 선형 회귀 모델 1: $\text{Rider ratings} = 3.8342 - 0.0917*(\text{Delivery minute} - 30)$
        - $R^{2}$ = 0.73
    - 선형 회귀 모델 2: $\text{Rider ratings} = 2.68 + 2.26* \text{I(Delivery less than 30 Minutes)} + 0.0016* \text{Rider's age} + 0.0087* \text{Multiple flag}$
        - $R^{2}$ = 0.98
    - 배달 시간의 효과 추정
        - 회귀 모델 1: 배달 시간 30분이후부터 1분씩 오를 때 -> 라이더 평점 0.09씩 감소
        - 회귀 모델 2: 배달 시간이 30분 미만인 그룹에 경우, 평균적으로 라이더 평점이 배달 시간이 30분 이상인 그룹보다 2.26점 높음, 라이더 나이, 여러배달 여부 효과 통제
    - 배달 시간에 따른 라이더 평점 예측
        - 배달 시간 40분 -> 평점 예측: 2.92
        - 배달 시간 37분 -> 평점 예측: 3.17
        - 배달 시간 30분 -> 평점 예측: 3.83 
        - 배달 시간 20분 -> 평점 예측: 4.75
        - 배달 시간 37분 -> 배달 시간 30분 (-23%), 평점 예측 3.17점 -> 3.83점 (+16%)
    - 목표 평점 도달을 위해 필요한 배달시간 예측
        - 목표 평점 4.5 -> 예측 배달 시간 22분
     
      

##### Limitations

- 각 라이더의 평균 배달 평점이 4.3 ~ 4.9 로 각 라이더 평균 배달 평점으로 낮은 평점의 배달의 분석이 어려움
- 긴 배달 시간의 병목을 규명하기 위한 데이터가 충분하지 않음 (픽업시간 파생변수 생성 가능, 배차대기시간 알 수 없음)
    - 배달 수요, 라이더 수요, 공급에대한 데이터 필요 
- 배달 주문 시간 또는 배달 픽업 시간의 결측치가 많아 제거 후 Complete Case Analysis가 가능하지만 분석의 오류 가능성이 있음
  

5) 비즈니스 제안
- 맑은 날씨, 낮은 교통체증의 배달시간 30분 미만으로 유지 
- 맑은 날씨, 낮은 교통체증의 배달시간 30분이상 병목구간 파악 (데이터 필요: 음식 준비 시간, 배차대기시간) 
- 맑은 날씨, 낮은 교통체증의 30분 미만의 배달시간 요인 분석 -> 30분 이상의 배달에 적용 추후 A/B Testing 후 배달시간 개선 

6) 사용 기술 
- Python (Pandas, statsmodel, LinearRegression, RandomForestRegressor)

7) 포트폴리오 한줄 요약
   
- 45000여 건의 배달 데이터를 분석하여 낮은 평점(3.0 이하)이 주로 맑은 날씨, 낮은 교통량, 22-24시 사이에 30분이상의 배달시간임을 규명하고,
배달 시간과 라이더 평점의 상관 관계를 모델링하여 배달 시간 (37분 -> 30분) 23% 단축시 라이더 평점 예측 (3.17점 -> 3.83점)(+16%) 개선 시나리오를 제시함 